In [66]:
import sys 
!{sys.executable} -m pip install json

ERROR: Could not find a version that satisfies the requirement json (from versions: none)
ERROR: No matching distribution found for json


In [2]:
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser

In [ ]:
from dotenv import load_dotenv
load_dotenv()
KEY=os.getenv("GROK_API_KEY")


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


In [6]:

llm=ChatGroq(model="llama-3.1-8b-instant",
            api_key=KEY,
            temperature=0.5,
            )

In [4]:
import os
import json
import pandas as pd
import traceback

In [7]:
from langchain_core.prompts import PromptTemplate
from langchain_core.callbacks import get_usage_metadata_callback
import PyPDF2

In [ ]:
TEMPLATE="""
Text:{text}
You are an expert MCQ maker. Given the above text, it is your job to \
create a quiz of {number} muiltiple choice questions for {subject} students in {tone} tone.
Make sure the questions are not repeated and check all the questions to be conforming the text as well.
Make sure to the format your response like RESPONSE_JSON below and use it as a guide.\
Ensure to makw{number} MCQs

{response_json}
"""

In [9]:
quiz_generation_prompt=PromptTemplate(
    input_variables=["text","number","subject","tone","response_json"],
    template=TEMPLATE
)

In [58]:
RESPONSE_JSON={
    "1":{
        "mcq":"multiple choice question",
        "options":{
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here:"
        },
        "correct": "correct_answere",
    },
    "2":{
        "mcq":"multiple choice question",
        "options":{
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here:"
        },
        "correct": "correct_answere",
    },
    "3":{
        "mcq":"multiple choice question",
        "options":{
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here:"
        },
        "correct": "correct_answere",
    },
    "4":{
        "mcq":"multiple choice question",
        "options":{
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here:"
        },
        "correct": "correct_answere",
    }
}

In [13]:
quiz_chain=quiz_generation_prompt|llm|StrOutputParser()

In [14]:
TEMPLATE2="""
You are an expert english grammarian and writer. Given a Multiple Choice Quiz for {subject} students.\
You need to evaluate the complexity of the question and give a complete analysis of the quiz.Only use at max  50 words for complexity.
if the quiz is not at per with the cognitive and analytical abilities of the students,\
update the quiz questions which needs to be chnaged and change the tone such that is perfectly fits the students ability.
Quiz_MCQs:
{quiz}

check from an expert English writer of the above quiz:

"""

In [15]:
quiz_evaluation_prompt=PromptTemplate(
    input_variables=["subject","quiz"],
    template=TEMPLATE2
)

In [16]:
review_chain=quiz_evaluation_prompt|llm|StrOutputParser()

In [17]:
from langchain_core.runnables import RunnablePassthrough

generate_evaluate_chain=(
    {"quiz":quiz_chain,"subject":lambda x: x["subject"]}|RunnablePassthrough.assign(review=review_chain)
)

In [18]:
file_path=r"C:\Users\hp\mcqgen1\data.txt"

In [22]:
with open(file_path,'r') as file:
    TEXT=file.read()

In [18]:
TEXT

'Machine learning is the subset of artificial intelligence (AI) focused on algorithms that can “learn” the patterns of training data and, subsequently, make accurate inferences about new data. This pattern recognition ability enables machine learning models to make decisions or predictions without explicit, hard-coded instructions.\n\nMachine learning has come to dominate the field of AI: it provides the backbone of most modern AI systems, from forecasting models to autonomous vehicles to large language models (LLMs) and other generative AI tools.\n\nThe central premise of machine learning (ML) is that if you optimize a model’s performance on a dataset of tasks that adequately resemble the real-world problems it will be used for—through a process called model training—the model can make accurate predictions on the new data it sees in its ultimate use case.\n\nTraining itself is simply a means to an end: generalization, the translation of strong performance on training data to useful re

In [19]:
import json
json.dumps(RESPONSE_JSON)

'{"1": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here:"}, "correct": "correct_answere"}, "2": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here:"}, "correct": "correct_answere"}, "3": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here:"}, "correct": "correct_answere"}}'

In [20]:
NUMBER=5
SUBJECT="machine learning"
TONE="simple"

In [59]:
with get_usage_metadata_callback() as cb:
    response=generate_evaluate_chain.invoke({
        
        "text":TEXT,
        "number":NUMBER,
        "subject":SUBJECT,
        "tone":TONE,
        "response_json":json.dumps(RESPONSE_JSON, indent=5)
    })

In [45]:
print(response)

{'quiz': '### RESPONSE_JSON\n{\n  "1": {\n    "mcq": "What is the primary goal of machine learning?",\n    "options": {\n      "a": "To make predictions on new data",\n      "b": "To optimize a model\'s performance on a dataset",\n      "c": "To generalize strong performance on training data to real-world scenarios",\n      "d": "To explicitly program a model with hard-coded instructions"\n    },\n    "correct": "c"\n  },\n  "2": {\n    "mcq": "What is the main difference between machine learning and traditional AI?",\n    "options": {\n      "a": "Machine learning is faster than traditional AI",\n      "b": "Machine learning requires more data than traditional AI",\n      "c": "Machine learning relies on explicitly defined algorithms, while traditional AI relies on implicit learning patterns",\n      "d": "Machine learning is less accurate than traditional AI"\n    },\n    "correct": "c"\n  },\n  "3": {\n    "mcq": "What is the term for the process of applying patterns learned from tr

In [63]:
quiz=response.get("quiz")
quiz2=json.dumps(quiz)


In [70]:
quiz3=print(json.loads(quiz2))
quiz3

### RESPONSE_JSON
{
    "1": {
        "mcq": "What is the primary goal of machine learning?",
        "options": {
            "a": "To make predictions on new data without explicit instructions",
            "b": "To optimize model performance on a specific dataset",
            "c": "To automate data analysis and apply learnings to tasks",
            "d": "To create expert systems with explicit logic"
        },
        "correct": "a"
    },
    "2": {
        "mcq": "What is the key difference between rules-based AI and machine learning?",
        "options": {
            "a": "Rules-based AI is more complex than machine learning",
            "b": "Machine learning requires explicit programming of logic",
            "c": "Rules-based AI requires manual definition of criteria, while machine learning learns from data",
            "d": "Machine learning is only used for simple tasks"
        },
        "correct": "c"
    },
    "3": {
        "mcq": "What is the term for the proce